# 📊 Data Exploration - TripTrove RAG

Notebook ini untuk eksplorasi data dari database MySQL dan PDF documents.

## Tujuan:
- Melihat struktur data tour packages
- Menganalisis konten PDF documents
- Memahami distribusi data
- Validasi kualitas data

In [ ]:
# Setup path
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Src path: {src_path}")

In [ ]:
# Import libraries
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os
from PyPDF2 import PdfReader
import matplotlib.pyplot as plt
import seaborn as sns

# Load environment variables
load_dotenv(project_root / '.env')

print("✅ Libraries loaded successfully!")

## 1. Database Exploration

In [ ]:
# Connect to database
db_config = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'triptrove_db')
}

conn = mysql.connector.connect(**db_config)
print("✅ Connected to database!")

In [ ]:
# Load tour packages
query = """
SELECT 
    id, name, description, price, discount_percent,
    duration_days, duration_nights, category, status,
    rating, total_reviews, created_at
FROM tour_packages
WHERE status = 'published'
"""

df_tours = pd.read_sql(query, conn)
print(f"📦 Total tour packages: {len(df_tours)}")
df_tours.head()

In [ ]:
# Data statistics
print("📊 Data Statistics:\n")
print(df_tours.describe())

In [ ]:
# Price distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df_tours['price'], bins=20, color='skyblue', edgecolor='black')
plt.xlabel('Price (Rp)')
plt.ylabel('Frequency')
plt.title('Price Distribution')

plt.subplot(1, 2, 2)
category_counts = df_tours['category'].value_counts()
plt.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%')
plt.title('Tour Categories')

plt.tight_layout()
plt.show()

In [ ]:
# Rating analysis
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_tours, x='category', y='rating')
plt.xticks(rotation=45)
plt.title('Rating by Category')
plt.tight_layout()
plt.show()

## 2. PDF Documents Exploration

In [ ]:
# List PDF files
pdf_dir = project_root / 'documents'
pdf_files = list(pdf_dir.glob('*.pdf'))

print(f"📄 Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"  - {pdf.name}")

In [ ]:
# Analyze PDF content
if pdf_files:
    for pdf_path in pdf_files:
        print(f"\n📖 Analyzing: {pdf_path.name}")
        print("=" * 50)
        
        reader = PdfReader(str(pdf_path))
        print(f"Total pages: {len(reader.pages)}")
        
        # Extract first page text
        first_page = reader.pages[0].extract_text()
        print(f"\nFirst 500 characters:")
        print(first_page[:500])
        print("...")
else:
    print("⚠️ No PDF files found")

## 3. Data Quality Check

In [ ]:
# Check for missing values
print("🔍 Missing Values:")
print(df_tours.isnull().sum())

In [ ]:
# Check for duplicates
duplicates = df_tours.duplicated(subset=['name']).sum()
print(f"\n🔍 Duplicate tour names: {duplicates}")

In [ ]:
# Price range by category
print("\n💰 Price Range by Category:")
print(df_tours.groupby('category')['price'].agg(['min', 'max', 'mean', 'count']))

In [ ]:
# Close connection
conn.close()
print("\n✅ Database connection closed")

## 📝 Summary

Gunakan cell di atas untuk:
- Memahami distribusi data tour packages
- Menganalisis konten PDF documents
- Validasi kualitas data sebelum digunakan untuk RAG
- Identifikasi data yang perlu diperbaiki